In [1]:
import os
from pathlib import Path

import pandas as pd

import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*ValueWarning.*")
warnings.filterwarnings("ignore", message=".*column_view.*")

force = False
anonymizer = True

root_dir = Path.cwd().parent.parent


def anondir(path: Path, prefix=root_dir) -> Path:
    """Anonymize a directory path by replacing user-specific parts with <root>."""
    if not anonymizer:
        return path
    path_str = str(path).replace(str(prefix), "<living-park>")
    return Path(path_str)


print(f"Running in root dir: {anondir(root_dir)}")

input_dir_sig = root_dir / "results" / "significant_digits"
assert input_dir_sig.exists(), (
    f"Input directory does not exist: {anondir(input_dir_sig)}"
)
print(f"Input directory: {anondir(input_dir_sig)}")

input_dir_std = root_dir / "results" / "std"
assert input_dir_std.exists(), (
    f"Input directory does not exist: {anondir(input_dir_std)}"
)
print(f"Input directory: {anondir(input_dir_std)}")

Running in root dir: <living-park>
Input directory: <living-park>/results/significant_digits
Input directory: <living-park>/results/std


## Significant digits

In [2]:
cortical_regions = [
    "bankssts",
    "caudalanteriorcingulate",
    "caudalmiddlefrontal",
    "cuneus",
    "entorhinal",
    "fusiform",
    "inferiorparietal",
    "inferiortemporal",
    "isthmuscingulate",
    "lateraloccipital",
    "lateralorbitofrontal",
    "lingual",
    "medialorbitofrontal",
    "middletemporal",
    "parahippocampal",
    "paracentral",
    "parsopercularis",
    "parsorbitalis",
    "parstriangularis",
    "pericalcarine",
    "postcentral",
    "posteriorcingulate",
    "precentral",
    "precuneus",
    "rostralanteriorcingulate",
    "rostralmiddlefrontal",
    "superiorfrontal",
    "superiorparietal",
    "superiortemporal",
    "supramarginal",
    "frontalpole",
    "temporalpole",
    "transversetemporal",
    "insula",
]

subcortical_regions = [
    "Left-Thalamus",
    "Left-Caudate",
    "Left-Putamen",
    "Left-Pallidum",
    "Left-Hippocampus",
    "Left-Amygdala",
    "Left-Accumbens-area",
    "Right-Thalamus",
    "Right-Caudate",
    "Right-Putamen",
    "Right-Pallidum",
    "Right-Hippocampus",
    "Right-Amygdala",
    "Right-Accumbens-area",
]

In [3]:
def get_sd(metric, hemisphere=None):
    """
    Load the significant digits data for a given metric.
    If hemisphere is specified, filter the data for that hemisphere.
    """
    num_filename = input_dir_sig / f"{metric}_num_significant_digits.parquet"
    df = pd.read_parquet(num_filename)
    # Average over regions
    group_by_cols = ["region"]
    if hemisphere:
        group_by_cols.append("hemisphere")
    df = (
        df.groupby(group_by_cols)["significant_digits"]
        .agg(lambda x: (x.mean().round(2), x.std().round(2)))
        .T
    ).reset_index(group_by_cols)
    # expand significant_digits into separate columns mean and std
    df[["mean", "std"]] = pd.DataFrame(
        df["significant_digits"].tolist(), index=df.index
    )
    df.drop(columns=["significant_digits"], inplace=True)
    return df

In [4]:
import pandas as pd

df_thickness = get_sd("thickness")
df_area = get_sd("area")
df_volume = get_sd("volume")


def get_sd_region(metric, region):
    df = get_sd(metric, hemisphere=True)
    df_region = df[df["region"] == region]
    values = {}
    for hemisphere in ["Left", "Right"]:
        mean = df_region[df_region["hemisphere"] == hemisphere]["mean"].values[0]
        std = df_region[df_region["hemisphere"] == hemisphere]["std"].values[0]
        values[hemisphere] = dict(mean=mean, std=std)
    return values


# display all rows
pd.set_option("display.max_rows", None)

skip_regions = []

for region in cortical_regions:
    thickness = get_sd_region("thickness", region)
    area = get_sd_region("area", region)
    volume = get_sd_region("volume", region)
    print(
        (
            rf"{region} & ${thickness['Left']['mean']:.02f} \pm {thickness['Left']['std']:.02f}$ & "
            rf"${thickness['Right']['mean']:.02f} \pm {thickness['Right']['std']:.02f}$ & "
            rf"${area['Left']['mean']:.02f} \pm {area['Left']['std']:.02f}$ & "
            rf"${area['Right']['mean']:.02f} \pm {area['Right']['std']:.02f}$ & "
            rf"${volume['Left']['mean']:.02f} \pm {volume['Left']['std']:.02f}$ & "
            rf"${volume['Right']['mean']:.02f} \pm {volume['Right']['std']:.02f}$ \\\\"
        )
    )

bankssts & $1.66 \pm 0.15$ & $1.70 \pm 0.13$ & $1.16 \pm 0.17$ & $1.22 \pm 0.12$ & $1.09 \pm 0.17$ & $1.14 \pm 0.12$ \\\\
caudalanteriorcingulate & $1.39 \pm 0.14$ & $1.40 \pm 0.14$ & $1.15 \pm 0.22$ & $1.19 \pm 0.17$ & $1.15 \pm 0.23$ & $1.21 \pm 0.19$ \\\\
caudalmiddlefrontal & $1.78 \pm 0.17$ & $1.78 \pm 0.18$ & $1.41 \pm 0.19$ & $1.32 \pm 0.21$ & $1.41 \pm 0.19$ & $1.31 \pm 0.21$ \\\\


cuneus & $1.53 \pm 0.19$ & $1.54 \pm 0.18$ & $1.34 \pm 0.14$ & $1.33 \pm 0.13$ & $1.33 \pm 0.14$ & $1.28 \pm 0.15$ \\\\
entorhinal & $1.22 \pm 0.23$ & $1.23 \pm 0.23$ & $0.83 \pm 0.18$ & $0.88 \pm 0.18$ & $0.80 \pm 0.19$ & $0.81 \pm 0.18$ \\\\
fusiform & $1.67 \pm 0.15$ & $1.71 \pm 0.15$ & $1.41 \pm 0.17$ & $1.44 \pm 0.19$ & $1.34 \pm 0.18$ & $1.38 \pm 0.20$ \\\\


inferiorparietal & $1.81 \pm 0.14$ & $1.83 \pm 0.12$ & $1.54 \pm 0.17$ & $1.60 \pm 0.19$ & $1.50 \pm 0.16$ & $1.57 \pm 0.16$ \\\\
inferiortemporal & $1.66 \pm 0.16$ & $1.71 \pm 0.15$ & $1.38 \pm 0.24$ & $1.39 \pm 0.20$ & $1.37 \pm 0.22$ & $1.41 \pm 0.18$ \\\\


isthmuscingulate & $1.46 \pm 0.11$ & $1.44 \pm 0.12$ & $1.27 \pm 0.14$ & $1.25 \pm 0.14$ & $1.27 \pm 0.13$ & $1.28 \pm 0.13$ \\\\
lateraloccipital & $1.76 \pm 0.17$ & $1.78 \pm 0.16$ & $1.58 \pm 0.15$ & $1.57 \pm 0.16$ & $1.50 \pm 0.15$ & $1.51 \pm 0.15$ \\\\
lateralorbitofrontal & $1.65 \pm 0.17$ & $1.52 \pm 0.15$ & $1.45 \pm 0.23$ & $0.95 \pm 0.13$ & $1.52 \pm 0.15$ & $1.11 \pm 0.13$ \\\\
lingual & $1.54 \pm 0.21$ & $1.52 \pm 0.21$ & $1.47 \pm 0.18$ & $1.46 \pm 0.17$ & $1.50 \pm 0.17$ & $1.50 \pm 0.17$ \\\\


medialorbitofrontal & $1.50 \pm 0.15$ & $1.54 \pm 0.15$ & $1.09 \pm 0.16$ & $1.16 \pm 0.14$ & $1.15 \pm 0.16$ & $1.21 \pm 0.12$ \\\\


...

parstriangularis & $1.68 \pm 0.17$ & $1.64 \pm 0.18$ & $1.33 \pm 0.15$ & $1.31 \pm 0.21$ & $1.31 \pm 0.15$ & $1.29 \pm 0.20$ \\\\


pericalcarine & $1.33 \pm 0.21$ & $1.31 \pm 0.22$ & $1.23 \pm 0.20$ & $1.22 \pm 0.21$ & $1.18 \pm 0.17$ & $1.18 \pm 0.16$ \\\\
postcentral & $1.84 \pm 0.23$ & $1.83 \pm 0.25$ & $1.68 \pm 0.22$ & $1.70 \pm 0.27$ & $1.65 \pm 0.18$ & $1.64 \pm 0.23$ \\\\
posteriorcingulate & $1.57 \pm 0.13$ & $1.56 \pm 0.14$ & $1.38 \pm 0.20$ & $1.35 \pm 0.21$ & $1.39 \pm 0.18$ & $1.39 \pm 0.22$ \\\\
precentral & $1.80 \pm 0.24$ & $1.78 \pm 0.27$ & $1.72 \pm 0.22$ & $1.65 \pm 0.26$ & $1.73 \pm 0.20$ & $1.68 \pm 0.25$ \\\\
precuneus & $1.83 \pm 0.12$ & $1.85 \pm 0.12$ & $1.67 \pm 0.19$ & $1.68 \pm 0.19$ & $1.62 \pm 0.16$ & $1.63 \pm 0.17$ \\\\


rostralanteriorcingulate & $1.35 \pm 0.13$ & $1.39 \pm 0.14$ & $1.00 \pm 0.16$ & $1.08 \pm 0.16$ & $1.11 \pm 0.18$ & $1.11 \pm 0.17$ \\\\


rostralmiddlefrontal & $1.78 \pm 0.18$ & $1.75 \pm 0.18$ & $1.46 \pm 0.23$ & $1.43 \pm 0.27$ & $1.50 \pm 0.20$ & $1.49 \pm 0.24$ \\\\
superiorfrontal & $1.88 \pm 0.17$ & $1.87 \pm 0.17$ & $1.62 \pm 0.22$ & $1.57 \pm 0.25$ & $1.65 \pm 0.19$ & $1.63 \pm 0.23$ \\\\
superiorparietal & $1.92 \pm 0.17$ & $1.94 \pm 0.16$ & $1.73 \pm 0.22$ & $1.67 \pm 0.25$ & $1.68 \pm 0.20$ & $1.61 \pm 0.24$ \\\\
superiortemporal & $1.84 \pm 0.16$ & $1.86 \pm 0.14$ & $1.58 \pm 0.21$ & $1.59 \pm 0.17$ & $1.53 \pm 0.20$ & $1.58 \pm 0.16$ \\\\
supramarginal & $1.84 \pm 0.14$ & $1.86 \pm 0.14$ & $1.58 \pm 0.19$ & $1.60 \pm 0.23$ & $1.57 \pm 0.18$ & $1.57 \pm 0.22$ \\\\


frontalpole & $1.27 \pm 0.23$ & $1.24 \pm 0.20$ & $0.94 \pm 0.11$ & $0.91 \pm 0.11$ & $0.89 \pm 0.16$ & $0.87 \pm 0.14$ \\\\
temporalpole & $1.24 \pm 0.25$ & $1.28 \pm 0.24$ & $0.94 \pm 0.16$ & $1.00 \pm 0.18$ & $0.87 \pm 0.20$ & $0.92 \pm 0.21$ \\\\
transversetemporal & $1.48 \pm 0.19$ & $1.47 \pm 0.18$ & $1.18 \pm 0.12$ & $1.13 \pm 0.11$ & $1.21 \pm 0.14$ & $1.15 \pm 0.12$ \\\\
insula & $1.47 \pm 0.15$ & $1.43 \pm 0.13$ & $1.13 \pm 0.17$ & $1.00 \pm 0.17$ & $1.29 \pm 0.16$ & $1.18 \pm 0.18$ \\\\


In [5]:
def get_sd_subcortical_volume(region):
    df = get_sd("subcortical_volume", hemisphere=None)
    df_region = df[df["region"] == region]
    mean = df_region["mean"].values[0]
    std = df_region["std"].values[0]
    return mean, std


for region in subcortical_regions:
    mean, std = get_sd_subcortical_volume(region)
    print(rf"{region} & ${mean:.02f} \pm {std:.02f}$ \\\\")

Left-Thalamus & $1.41 \pm 0.21$ \\\\
Left-Caudate & $1.58 \pm 0.20$ \\\\
Left-Putamen & $1.49 \pm 0.22$ \\\\
Left-Pallidum & $1.25 \pm 0.19$ \\\\
Left-Hippocampus & $1.48 \pm 0.17$ \\\\
Left-Amygdala & $1.13 \pm 0.15$ \\\\


Left-Accumbens-area & $0.88 \pm 0.16$ \\\\
Right-Thalamus & $1.42 \pm 0.20$ \\\\
Right-Caudate & $1.52 \pm 0.24$ \\\\
Right-Putamen & $1.51 \pm 0.25$ \\\\
Right-Pallidum & $1.23 \pm 0.19$ \\\\
Right-Hippocampus & $1.55 \pm 0.17$ \\\\
Right-Amygdala & $1.23 \pm 0.17$ \\\\
Right-Accumbens-area & $0.99 \pm 0.15$ \\\\


## Standard deviation

In [6]:
def get_std(metric, hemisphere=None):
    """
    Load the standard deviation data for a given metric.
    If hemisphere is specified, filter the data for that hemisphere.
    """
    num_filename = input_dir_std / f"{metric}_num_std.parquet"
    df = pd.read_parquet(num_filename)
    df.rename(columns={"std": "stddev"}, inplace=True)
    # Average over regions
    group_by_cols = ["region"]
    if hemisphere:
        group_by_cols.append("hemisphere")
    df = (
        df.groupby(group_by_cols)["stddev"]
        .agg(lambda x: (x.mean().round(2), x.std().round(2)))
        .T
    ).reset_index(group_by_cols)
    # expand std into separate columns mean and std
    df[["mean", "std"]] = pd.DataFrame(df["stddev"].tolist(), index=df.index)
    df.drop(columns=["stddev"], inplace=True)
    return df

In [7]:
import pandas as pd

df_thickness = get_std("thickness")
df_area = get_std("area")
df_volume = get_std("volume")


def get_std_region(metric, region):
    """Get statistics for a specific region and metric"""
    df = get_std(metric, hemisphere=True)
    df_region = df[df["region"] == region]
    values = {}
    for hemisphere in ["Left", "Right"]:
        hemisphere_data = df_region[df_region["hemisphere"] == hemisphere]
        if not hemisphere_data.empty:
            mean = hemisphere_data["mean"].values[0]
            std = hemisphere_data["std"].values[0]
            values[hemisphere] = dict(mean=mean, std=std)
        else:
            # Handle case where no data exists for this hemisphere
            values[hemisphere] = dict(mean=0.0, std=0.0)
    return values


def get_max_digits_before_decimal(values):
    """
    Get the maximum number of digits before decimal point in a list of values.

    Args:
        values: List of numerical values

    Returns:
        int: Maximum number of digits before decimal point
    """
    if not values:
        return 0

    max_digits = 0
    for value in values:
        if value > 0:
            # Count digits before decimal point
            digits = len(str(int(value)))
            max_digits = max(max_digits, digits)

    return max_digits


def get_leading_zeros(value, max_digits):
    """
    Returns the appropriate LaTeX leading zeros (\\0) for alignment.

    Args:
        value: The numerical value
        max_digits: Maximum number of digits before decimal point in the column

    Returns:
        str: LaTeX leading zeros string
    """
    if value <= 0:
        return ""

    # Count digits in current value
    current_digits = len(str(int(value)))

    # Calculate how many leading zeros needed
    zeros_needed = max_digits - current_digits

    # Return LaTeX leading zeros
    return "\\0" * zeros_needed


def collect_all_values_for_metric(all_regions_data, metric_type):
    """
    Collect all values (mean or std) for a specific metric across all regions and hemispheres.

    Args:
        all_regions_data: Dictionary with region data for thickness, area, volume
        metric_type: 'mean' or 'std'

    Returns:
        dict: Dictionary with 'thickness', 'area', 'volume' keys containing lists of values
    """
    collected = {"thickness": [], "area": [], "volume": []}

    for region in all_regions_data:
        for metric in ["thickness", "area", "volume"]:
            for hemisphere in ["Left", "Right"]:
                if hemisphere in all_regions_data[region][metric]:
                    value = all_regions_data[region][metric][hemisphere][metric_type]
                    collected[metric].append(value)

    return collected


# Display all rows
pd.set_option("display.max_rows", None)

# First pass: collect all data for all regions to determine max digits for alignment
all_regions_data = {}
for region in cortical_regions:
    thickness = get_std_region("thickness", region)
    area = get_std_region("area", region)
    volume = get_std_region("volume", region)

    all_regions_data[region] = {"thickness": thickness, "area": area, "volume": volume}

# Calculate maximum digits for each metric and statistic type
mean_values = collect_all_values_for_metric(all_regions_data, "mean")
std_values = collect_all_values_for_metric(all_regions_data, "std")

# Get max digits for each metric
thickness_mean_max_digits = get_max_digits_before_decimal(mean_values["thickness"])
thickness_std_max_digits = get_max_digits_before_decimal(std_values["thickness"])
area_mean_max_digits = get_max_digits_before_decimal(mean_values["area"])
area_std_max_digits = get_max_digits_before_decimal(std_values["area"])
volume_mean_max_digits = get_max_digits_before_decimal(mean_values["volume"])
volume_std_max_digits = get_max_digits_before_decimal(std_values["volume"])

# Second pass: format and print with proper leading zeros
for region in cortical_regions:
    thickness = all_regions_data[region]["thickness"]
    area = all_regions_data[region]["area"]
    volume = all_regions_data[region]["volume"]

    # Get leading zeros for each value
    thickness_left_mean_zeros = get_leading_zeros(
        thickness["Left"]["mean"], thickness_mean_max_digits
    )
    thickness_left_std_zeros = get_leading_zeros(
        thickness["Left"]["std"], thickness_std_max_digits
    )
    thickness_right_mean_zeros = get_leading_zeros(
        thickness["Right"]["mean"], thickness_mean_max_digits
    )
    thickness_right_std_zeros = get_leading_zeros(
        thickness["Right"]["std"], thickness_std_max_digits
    )

    area_left_mean_zeros = get_leading_zeros(area["Left"]["mean"], area_mean_max_digits)
    area_left_std_zeros = get_leading_zeros(area["Left"]["std"], area_std_max_digits)
    area_right_mean_zeros = get_leading_zeros(
        area["Right"]["mean"], area_mean_max_digits
    )
    area_right_std_zeros = get_leading_zeros(area["Right"]["std"], area_std_max_digits)

    volume_left_mean_zeros = get_leading_zeros(
        volume["Left"]["mean"], volume_mean_max_digits
    )
    volume_left_std_zeros = get_leading_zeros(
        volume["Left"]["std"], volume_std_max_digits
    )
    volume_right_mean_zeros = get_leading_zeros(
        volume["Right"]["mean"], volume_mean_max_digits
    )
    volume_right_std_zeros = get_leading_zeros(
        volume["Right"]["std"], volume_std_max_digits
    )

    # Format strings with leading zeros
    thickness_left_str = f"{thickness_left_mean_zeros}{thickness['Left']['mean']:.2f} \\pm {thickness_left_std_zeros}{thickness['Left']['std']:.2f}"
    thickness_right_str = f"{thickness_right_mean_zeros}{thickness['Right']['mean']:.2f} \\pm {thickness_right_std_zeros}{thickness['Right']['std']:.2f}"

    area_left_str = f"{area_left_mean_zeros}{area['Left']['mean']:.2f} \\pm {area_left_std_zeros}{area['Left']['std']:.2f}"
    area_right_str = f"{area_right_mean_zeros}{area['Right']['mean']:.2f} \\pm {area_right_std_zeros}{area['Right']['std']:.2f}"

    volume_left_str = f"{volume_left_mean_zeros}{volume['Left']['mean']:.2f} \\pm {volume_left_std_zeros}{volume['Left']['std']:.2f}"
    volume_right_str = f"{volume_right_mean_zeros}{volume['Right']['mean']:.2f} \\pm {volume_right_std_zeros}{volume['Right']['std']:.2f}"

    # Print LaTeX table row
    print(
        rf"{region} & ${thickness_left_str}$ & ${thickness_right_str}$ & "
        rf"${area_left_str}$ & ${area_right_str}$ & "
        rf"${volume_left_str}$ & ${volume_right_str}$ \\\\"
    )

bankssts & $0.02 \pm 0.01$ & $0.02 \pm 0.01$ & $\027.27 \pm \012.31$ & $\021.23 \pm \0\06.77$ & $\074.82 \pm \033.80$ & $\059.56 \pm \019.75$ \\\\
caudalanteriorcingulate & $0.04 \pm 0.01$ & $0.04 \pm 0.01$ & $\019.30 \pm \012.89$ & $\019.96 \pm \010.26$ & $\049.88 \pm \035.51$ & $\049.51 \pm \029.83$ \\\\
caudalmiddlefrontal & $0.02 \pm 0.01$ & $0.02 \pm 0.01$ & $\035.84 \pm \025.90$ & $\043.79 \pm \040.86$ & $\097.03 \pm \075.10$ & $118.40 \pm 103.50$ \\\\
cuneus & $0.02 \pm 0.01$ & $0.02 \pm 0.01$ & $\028.10 \pm \011.52$ & $\030.55 \pm \011.58$ & $\059.82 \pm \025.31$ & $\074.10 \pm \033.48$ \\\\
entorhinal & $0.08 \pm 0.05$ & $0.08 \pm 0.05$ & $\026.61 \pm \014.27$ & $\021.77 \pm \010.84$ & $123.91 \pm \071.51$ & $113.59 \pm \054.35$ \\\\
fusiform & $0.02 \pm 0.01$ & $0.02 \pm 0.01$ & $\049.89 \pm \024.61$ & $\046.46 \pm \025.84$ & $180.70 \pm \089.05$ & $166.81 \pm 100.45$ \\\\
inferiorparietal & $0.01 \pm 0.01$ & $0.01 \pm 0.00$ & $\051.59 \pm \028.04$ & $\056.02 \pm \034.37$ & $

In [8]:
def get_std_subcortical_volume(region):
    df = get_std("subcortical_volume", hemisphere=None)
    df_region = df[df["region"] == region]
    mean = df_region["mean"].values[0]
    std = df_region["std"].values[0]
    return mean, std


for region in subcortical_regions:
    mean, std = get_std_subcortical_volume(region)
    print(rf"{region} & ${mean:.02f} \pm {std:.02f}$ \\\\")

Left-Thalamus & $121.72 \pm 68.93$ \\\\
Left-Caudate & $37.92 \pm 24.01$ \\\\
Left-Putamen & $65.18 \pm 46.06$ \\\\
Left-Pallidum & $47.39 \pm 24.91$ \\\\
Left-Hippocampus & $55.50 \pm 39.50$ \\\\
Left-Amygdala & $48.19 \pm 19.29$ \\\\
Left-Accumbens-area & $24.16 \pm 8.70$ \\\\
Right-Thalamus & $118.52 \pm 70.08$ \\\\
Right-Caudate & $48.78 \pm 44.28$ \\\\
Right-Putamen & $67.42 \pm 70.59$ \\\\
Right-Pallidum & $48.00 \pm 27.88$ \\\\
Right-Hippocampus & $48.26 \pm 29.47$ \\\\
Right-Amygdala & $41.77 \pm 18.52$ \\\\
Right-Accumbens-area & $20.74 \pm 7.60$ \\\\
